In [ ]:
# Magic code for skipping cells execution
from IPython.core.magic import register_cell_magic

@register_cell_magic
def skip(line, cell):
    return

In [5]:
# Adapted from https://github.com/KULeuven-COSIC/SimpleLink-FI/blob/main/notebooks/5_ChipSHOUTER-PicoEMP.ipynb
# A very basic class to interact with the PicoEMP

import serial
import time

class PicoEMP:
    def __init__(self, port='/dev/picoEMP'):
        self.pico = serial.Serial(port, 115200)
        
        self.pico.write(b'\r\n')
        time.sleep(0.1)
        ret = self.pico.read(self.pico.in_waiting)
        
        if b'PicoEMP Commands' in ret:
            print('Connected to PicoEMP!')
        else:
            raise OSError('Could not connect to PicoEMP :(')
        
    def disconnect(self):
        self.pico.close()
        time.sleep(1)
        print('Disconnected from PicoEMP!')

    def disable_timeout(self):
        self.pico.write(b'disable_timeout\r\n')
        time.sleep(1)
        assert b'Timeout disabled!' in self.pico.read(self.pico.in_waiting)

    def arm(self):
        self.pico.write(b'arm\r\n')
        time.sleep(1)
        assert b'Device armed' in self.pico.read(self.pico.in_waiting)

    def disarm(self):
        self.pico.write(b'disarm\r\n')
        time.sleep(1)
        assert b'Device disarmed!' in self.pico.read(self.pico.in_waiting)

    def set_fast_trigger(self):
        self.pico.write(b'fast_trigger\r\n')
        ## The pico then waits to see the trigger

    def configure_fast_trigger(self, delay_cycles, pulse_cycles):
        self.pico.write(b'fast_trigger_configure\r\n' + f'{delay_cycles}\r\n'.encode('utf-8') + f'{pulse_cycles}\r\n'.encode('utf-8'))

    def internal_hvp(self):
        self.pico.write(b'internal_hvp\r\n')
        time.sleep(1)
        assert b'Internal HVP mode active' in self.pico.read(self.pico.in_waiting)

    def external_hvp(self):
        self.pico.write(b'external_hvp\r\n')
        time.sleep(1)
        assert b'External HVP mode active' in self.pico.read(self.pico.in_waiting)

    def reset_target(self):
        self.pico.write(b't\r\n')
        time.sleep(0.1)
        self.pico.write(b't\r\n')
        
    def print_status(self):
        self.pico.write(b'status\r\n')
        time.sleep(1)
        print(self.pico.read(self.pico.in_waiting).decode('utf-8'))

    def setup_internal_control(self):
        self.disable_timeout()
        self.internal_hvp()
        self.print_status()
    
    def setup_external_control(self):
        self.disable_timeout()
        self.external_hvp()
        self.print_status()
